![](./drawings/accounts_merge_solution_00.svg)

In [ ]:
class UnionFind:

    def __init__(self, totalNumberOfGraphNodes: int):
        self.nodeToParentMap = dict()
        self.nodeToSubTreeSizeMap = dict()

        for node in range(totalNumberOfGraphNodes):
            self.nodeToParentMap[node] = node
            self.nodeToSubTreeSizeMap[node] = 1

    def find(self, node: int) -> int:
        currentNode = node
        while currentNode != self.nodeToParentMap[currentNode]:
            currentParentNode = self.nodeToParentMap[currentNode]
            currentGrandparentNode = self.nodeToParentMap[currentParentNode]

            self.nodeToParentMap[currentNode] = currentGrandparentNode
            currentNode = currentParentNode
        return currentNode

    def union(self, node1: int, node2: int) -> bool:
        root1 = self.find(node1)
        root2 = self.find(node2)

        if root1 == root2:
            return False

        if self.nodeToSubTreeSizeMap[root1] > self.nodeToSubTreeSizeMap[root2]:
            self.nodeToParentMap[root2] = root1
            self.nodeToSubTreeSizeMap[root1] += self.nodeToSubTreeSizeMap[root2]
        else:
            self.nodeToParentMap[root1] = root2
            self.nodeToSubTreeSizeMap[root2] += self.nodeToSubTreeSizeMap[root1]

        return True

class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        unionFind = UnionFind(len(accounts))
        emailToAccountIndexMap = dict()
        for accountIndex in range(len(accounts)):
            for emailIndex in range(1, len(accounts[accountIndex])):
                email = accounts[accountIndex][emailIndex]
                if email not in emailToAccountIndexMap:
                    emailToAccountIndexMap[email] = accountIndex
                else:
                    unionFind.union(emailToAccountIndexMap[email], accountIndex)
        
        groupRootAccountIndexToEmailsMap = dict()
        for email, accountIndex in emailToAccountIndexMap.items():
            groupRootAccountIndex = unionFind.find(accountIndex)
            if groupRootAccountIndex not in groupRootAccountIndexToEmailsMap:
                groupRootAccountIndexToEmailsMap[groupRootAccountIndex] = [email]
            else:
                groupRootAccountIndexToEmailsMap[groupRootAccountIndex].append(email)

        mergedAccounts = []
        for groupRootAccountIndex, emails in groupRootAccountIndexToEmailsMap.items():
            accountName = accounts[groupRootAccountIndex][0]
            sortedEmails = sorted(emails)
            mergedAccounts.append([accountName] + sortedEmails)
        
        return mergedAccounts

![](./drawings/accounts_merge_solution_02.svg)

In [ ]:
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        n = len(accounts)
        emailIdx = {} # email -> id
        emails = [] # set of emails of all accounts
        emailToAcc = {} # email_index -> account_Id

        m = 0
        for accId, a in enumerate(accounts):
            for i in range(1, len(a)):
                email = a[i]
                if email in emailIdx:
                    continue
                emails.append(email)
                emailIdx[email] = m
                emailToAcc[m] = accId
                m += 1

        adj = [[] for _ in range(m)]
        for a in accounts:
            for i in range(2, len(a)):
                id1 = emailIdx[a[i]]
                id2 = emailIdx[a[i - 1]]
                adj[id1].append(id2)
                adj[id2].append(id1)

        emailGroup = defaultdict(list) # index of acc -> list of emails
        visited = [False] * m
        def dfs(node, accId):
            visited[node] = True
            emailGroup[accId].append(emails[node])
            for nei in adj[node]:
                if not visited[nei]:
                    dfs(nei, accId)

        for i in range(m):
            if not visited[i]:
                dfs(i, emailToAcc[i])

        res = []
        for accId in emailGroup:
            name = accounts[accId][0]
            res.append([name] + sorted(emailGroup[accId]))

        return res

![](./drawings/accounts_merge_solution_02.svg)

In [ ]:
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        n = len(accounts)
        emailIdx = {} # email -> id
        emails = [] # set of emails of all accounts
        emailToAcc = {} # email_index -> account_Id

        m = 0
        for accId, a in enumerate(accounts):
            for i in range(1, len(a)):
                email = a[i]
                if email in emailIdx:
                    continue
                emails.append(email)
                emailIdx[email] = m
                emailToAcc[m] = accId
                m += 1

        adj = [[] for _ in range(m)]
        for a in accounts:
            for i in range(2, len(a)):
                id1 = emailIdx[a[i]]
                id2 = emailIdx[a[i - 1]]
                adj[id1].append(id2)
                adj[id2].append(id1)

        emailGroup = defaultdict(list) # index of acc -> list of emails
        visited = [False] * m

        def bfs(start, accId):
            queue = deque([start])
            visited[start] = True
            while queue:
                node = queue.popleft()
                emailGroup[accId].append(emails[node])
                for nei in adj[node]:
                    if not visited[nei]:
                        visited[nei] = True
                        queue.append(nei)

        for i in range(m):
            if not visited[i]:
                bfs(i, emailToAcc[i])

        res = []
        for accId in emailGroup:
            name = accounts[accId][0]
            res.append([name] + sorted(emailGroup[accId]))

        return res

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.par = [i for i in range(n)]
        self.rank = [1] * n

    def find(self, x):
        while x != self.par[x]:
            self.par[x] = self.par[self.par[x]]
            x = self.par[x]
        return x

    def union(self, x1, x2):
        p1, p2 = self.find(x1), self.find(x2)
        if p1 == p2:
            return False
        if self.rank[p1] > self.rank[p2]:
            self.par[p2] = p1
            self.rank[p1] += self.rank[p2]
        else:
            self.par[p1] = p2
            self.rank[p2] += self.rank[p1]
        return True

class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        uf = UnionFind(len(accounts))
        emailToAcc = {}  # email -> index of acc

        for i, a in enumerate(accounts):
            for e in a[1:]:
                if e in emailToAcc:
                    uf.union(i, emailToAcc[e])
                else:
                    emailToAcc[e] = i

        emailGroup = defaultdict(list)  # index of acc -> list of emails
        for e, i in emailToAcc.items():
            leader = uf.find(i)
            emailGroup[leader].append(e)

        res = []
        for i, emails in emailGroup.items():
            name = accounts[i][0]
            res.append([name] + sorted(emailGroup[i]))
        return res